<a href="https://colab.research.google.com/github/JoaoVitorCoelhoG/Aprendizado-Profundo/blob/main/cm204_lab7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Instituto Tecnológico de Aeronáutica – ITA**

**Aprendizado Profundo - CM-204**

**Professores:**

Marcos Ricardo Omena de Albuquerque Maximo

Gabriel Adriano de Melo

Claude Code

---

# Laboratório 7 - Mecanismos de Atenção

**Observação**:
- **NÃO** exclua células, pois isto pode gerar problemas com o autocorretor!
- Para editar este notebook, salve uma cópia no seu Drive em ``File > Save a copy in Drive``.
- Neste laboratório, é aconselhável utilizar a GPU do Colab para o treinamento da rede. Para usar a GPU, faça o seguinte:
  - Na aba superior, selecione o menu "*Runtime*" ("Ambiente de execução");
  - Clique em "*Change runtime type*" ("Alterar tipo de ambiente de execução");
  - Selecione "GPU" na seção "*Hardware accelerator*" ("Acelerador de *hardware*") e salve.

  Verifique se a GPU está disponível executando o seguinte comando no notebook:

In [ ]:
!nvidia-smi


Neste laboratório, você vai implementar mecanismos de atenção, incluindo um Atenção Convolucional e uma cabeça Transformer.

Este laboratório foi originalmente criado por Gabriel Melo. Marcos Maximo fez melhorias. Nosso grande amigo `Claude Code` deu várias dicas de como implementar o Transformer para o TinyShakespeare.

In [ ]:
!pip install torch opencv-python numpy matplotlib tqdm

In [ ]:
import cv2
import numpy as np
from PIL import Image
from tqdm import tqdm
from collections import OrderedDict
from matplotlib import pyplot as plt

import torch
import torch.nn as nn

## 1. Convolutional Attention

Para imagens, temos a chamada atenção espacial. Ela é usada em muitos modelos convolucionais do estado da arte:

$\textbf{M}_{s}\left(F\right) = \sigma\left(f^{7x7}\left(\left[\text{AvgPool}\left(F\right);\text{MaxPool}\left(F\right)\right]\right)\right)$

$\textbf{M}_{s}\left(F\right) = \sigma\left(f^{7x7}\left(\left[\mathbf{F}^{s}_{avg};\mathbf{F}^{s}_{max} \right]\right)\right)$

Implemente a função abaixo, que está relacionada ao módulo de atenção espacial (SAM) do modelo CBAM de [Woo et. al (2018)](https://arxiv.org/abs/1807.06521).

Use as seguintes operações:
- `torch.cat`: concatena dois ou mais tensores;
- `torch.max`: calcula o valor máximo ao longo de uma dimensão;
- `torch.mean`: calcula a média ao longo de uma dimensão;
- `torch.sigmoid`: aplica a função de ativação sigmoide a cada elemento do tensor (elemento a elemento).

Verifique a documentação do Pytorch para cada função para entender melhor como usá-las.
Ao usar `torch.max` e `torch.mean`, você pode alterar o formato do tensor. Para corrigir isso, você pode usar `.unsqueeze`/`.view`/`.reshape`/`tensor[:, None, :, :]` ou pode usar o parâmetro `keepdim` como `True` para manter a dimensão.

Para multiplicar dois tensores elemento a elemento, use `*` ou `torch.multiply`.

Para executar a operação convolucional, use-a como uma chamada de função, isto é, como `operation(input)`.
<details><summary><b>---Dica---</b></summary>
<p>
No SAM, os valores médio e máximo são calculados ao longo dos canais. Em um tensor de formato (N, C, H, W), os canais estão na dimensão 1 (C).
</p>
</details>

In [ ]:
def spatial_attention(convolutional_operation: nn.Module, features: torch.tensor) -> torch.tensor:
    """
    Implements the spatial attention module (SAM) by appplying an attention map to the
    features in a multiplicative form. Obs.: concatenates the Avg before the Max.

    :param convolutional_operation: convolutional operation (using a filter 𝑓) which
                                         receives an image with two channels as input
                                         and returns an image with a single channel.
    :param features: tensor of shape (N, C, H, W), where N is the number of batches,
                             C is the number of channels, H is the height, and W is the width.
    return: tensor of shape (N, C, H, W), the resulting tensor after the spatial attention operation.
    """
    raise NotImplementedError() # Delete this line
    return result

In [ ]:
conv = nn.Conv2d(in_channels=2, out_channels=1, kernel_size=3, stride=1, padding=1)
conv.load_state_dict(OrderedDict([('weight', torch.tensor([[[ [-0.0165, -0.0119, -0.1987],
                                                              [ 0.1247, -0.0875,  0.0907],
                                                              [-0.1658,  0.2204,  0.1959]],
                                                             [[-0.0925, -0.1122, -0.2284],
                                                              [ 0.0141,  0.0105, -0.0061],
                                                              [ 0.0526,  0.1581, -0.1757]] ]])),
                                  ('bias', torch.tensor([0.0675]))]))
features = torch.tensor([[[[-0.0165, -0.0119, -0.1987, 1, 1, 1, 1],
                           [ 0.1247, -0.0875,  0.0907, 1, 1, 1, 1],
                           [ 0.1247, -0.0875,  0.0907, 1, 1, 1, 1],
                           [ 0.1247, -0.0875,  0.0907, 1, 1, 1, 1],
                           [-0.1658,  0.2204,  0.1959, 1, 1, 1, 1]],
                          [[-0.0165, -0.0119, -0.1987, 1, 1, 1, 1],
                           [ 0.1247, -0.0875,  0.0907, 1, 1, 1, 1],
                           [ 0.1247, -0.0875,  0.0907, 1, 1, 1, 1],
                           [ 0.1247, -0.0875,  0.0907, 1, 1, 1, 1],
                           [-0.1658,  0.2204,  0.1959, 1, 1, 1, 1]],
                          [[-0.0165, -0.0119, -0.1987, 1, 1, 1, 1],
                           [ 0.1247, -0.0875,  0.0907, 1, 1, 1, 1],
                           [ 0.1247, -0.0875,  0.0907, 1, 1, 1, 1],
                           [ 0.1247, -0.0875,  0.0907, 1, 1, 1, 1],
                           [-0.1658,  0.2204,  0.1959, 1, 1, 1, 1]]
                        ]])
result = spatial_attention(conv, features)
assert result.requires_grad
assert torch.norm(result - torch.tensor(
       [[[[-0.0087, -0.0060, -0.1107,  0.6073,  0.6223,  0.6223,  0.5974],
          [ 0.0656, -0.0469,  0.0407,  0.4866,  0.4599,  0.4599,  0.5403],
          [ 0.0660, -0.0440,  0.0401,  0.4788,  0.4599,  0.4599,  0.5403],
          [ 0.0628, -0.0473,  0.0402,  0.4758,  0.4599,  0.4599,  0.5403],
          [-0.0879,  0.1103,  0.0852,  0.3873,  0.3902,  0.3902,  0.4741]],

         [[-0.0087, -0.0060, -0.1107,  0.6073,  0.6223,  0.6223,  0.5974],
          [ 0.0656, -0.0469,  0.0407,  0.4866,  0.4599,  0.4599,  0.5403],
          [ 0.0660, -0.0440,  0.0401,  0.4788,  0.4599,  0.4599,  0.5403],
          [ 0.0628, -0.0473,  0.0402,  0.4758,  0.4599,  0.4599,  0.5403],
          [-0.0879,  0.1103,  0.0852,  0.3873,  0.3902,  0.3902,  0.4741]],

         [[-0.0087, -0.0060, -0.1107,  0.6073,  0.6223,  0.6223,  0.5974],
          [ 0.0656, -0.0469,  0.0407,  0.4866,  0.4599,  0.4599,  0.5403],
          [ 0.0660, -0.0440,  0.0401,  0.4788,  0.4599,  0.4599,  0.5403],
          [ 0.0628, -0.0473,  0.0402,  0.4758,  0.4599,  0.4599,  0.5403],
          [-0.0879,  0.1103,  0.0852,  0.3873,  0.3902,  0.3902,  0.4741]]]])).item() < 1e-3

## 2. Transformers

Um modelo de atenção inicialmente proposto para Processamento de Linguagem Natural (PLN) para permitir operações altamente paralelizáveis e facilitar o treinamento de sequências longas. Ele já foi adaptado para Visão Computacional por meio do Vision Transformer (ViT), que essencialmente trata a imagem como uma sequência de patches.

![Transformer Architecture](https://lilianweng.github.io/posts/2018-06-24-attention/transformer.png)

O Transformer foi inicialmente proposto por [Vaswani et. al (2017)](https://arxiv.org/abs/1706.03762v5), seguindo as equações abaixo:

$\text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}(\frac{\mathbf{Q}\mathbf{K}^\top}{\sqrt{n}})\mathbf{V}$

$\begin{aligned}
\text{MultiHead}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) &= [\text{head}_1; \dots; \text{head}_h]\mathbf{W}^O \\
\text{where head}_i &= \text{Attention}(\mathbf{Q}\mathbf{W}^Q_i, \mathbf{K}\mathbf{W}^K_i, \mathbf{V}\mathbf{W}^V_i)
\end{aligned}$

Implemente a cabeça de atenção do Transformer.

Use a função `.transpose(dim1, dim2)` para transpor as dimensões 1 e 2 de um tensor. Para fazer uma multiplicação de matrizes regular nas duas últimas dimensões de um tensor, use o operador `@` ou a função `torch.matmul`. Para obter as dimensões de um tensor, use `.shape ` ou `.size()`.
<details><summary><b>---Dica---</b></summary>
<p>
Para usar as operações de projeção, use-as como funções, de forma semelhante ao que você fez no exercício anterior.

Use `tensor.transpose(-1, -2)` para transpor as duas últimas dimensões de um tensor.
</p>
</details>

In [ ]:
def transformer_head(q: torch.tensor, k: torch.tensor, v: torch.tensor,
                     proj_q: nn.Module, proj_k: nn.Module, proj_v: nn.Module,
                     reproj: nn.Module, softmax: nn.Module):
    """
    Implements the head of the Transformer architecture given by equation
    (softmax((q' @ k'.T) / sqrt(D_k)) @ v')' where ' represents the operation of linear
    projection for each tensor on its vector space.

    :param q: query tensor of shape (N, L, D).
    :param k: key tensor of shape (N, L, D).
    :param v: value tensor of shape (N, L, D).
    :param proj_q: operation that does a linear projection of a tensor (N, L, D) on
                   H vector spaces of dimension D_k, resulting in a tensor (N, H, L, D_k).
    :param proj_k: operation that does a linear projection of a tensor (N, L, D) on
                   H vector spaces of dimension D_k, resulting in a tensor (N, H, L, D_k).
    :param proj_v: operation that does a linear projection of a tensor (N, L, D) on
                   H vector spaces of dimension D_v, resulting in a tensor (N, H, L, D_v).
    :param reproj: operation that does a concatenation and linear projection of a tensor
                   (N, H, L, D_v) of dimension E2, resulting in a tensor (N, L, D).
    :param softmax: softmax operation over the tensor last dimension.
    :return: tensor of shape (N, L, D), final result of the attention module.
    """
    raise NotImplementedError() # Delete this line
    return result

class Projection(nn.Linear):
    """
    Linear projection layer for the Transformer architecture.
    """
    def __init__(self, dim_in, num_projs, dim_out, **kwargs):
        super().__init__(dim_in, num_projs * dim_out, **kwargs)
        self.num_projs = num_projs

    def forward(self, x):
        x_flat = super().forward(x)
        return x_flat.reshape(*x_flat.shape[:2], self.num_projs, -1).transpose(1, 2)

class ReProjection(nn.Linear):
    """
    Re-projection layer for the Transformer architecture.
    """
    def __init__(self, dim_in, num_projs, dim_out, **kwargs):
        super().__init__(dim_in * num_projs, dim_out, **kwargs)
        self.num_projs = num_projs

    def forward(self, x):
        x_flat = x.transpose(1, 2).contiguous()
        return super().forward(x_flat.reshape(*x_flat.shape[:2], -1))

In [ ]:
softmax = nn.Softmax(dim=-1)
proj_q = Projection(4, 2, 3)
proj_k = Projection(4, 2, 3)
proj_v = Projection(4, 2, 5)
reproj = ReProjection(5, 2, 4)
proj_q.load_state_dict(OrderedDict([('weight', torch.tensor([[-0.2903,  0.1144, -0.2388,  0.3808],
                      [ 0.4571, -0.1722, -0.3059,  0.0529],
                      [-0.0764,  0.4670, -0.2218, -0.3888],
                      [ 0.4820, -0.0895,  0.0496, -0.4707],
                      [ 0.0030, -0.0348,  0.4132,  0.3539],
                      [-0.2953,  0.2528,  0.2744, -0.0833]])),
    ('bias', torch.tensor([-0.2345, -0.0744,  0.1075, -0.1458, -0.4157, -0.3114]))]))
proj_k.load_state_dict(OrderedDict([('weight', torch.tensor([[ 0.2761, -0.4321,  0.3839,  0.1454],
                      [-0.3980,  0.1969, -0.4166, -0.2317],
                      [-0.1690, -0.1395,  0.3167, -0.3027],
                      [-0.1422, -0.2583, -0.4430, -0.0448],
                      [ 0.4761,  0.1354,  0.1436,  0.3219],
                      [ 0.3956,  0.0885,  0.2519,  0.1227]])),
    ('bias', torch.tensor([0.4507, 0.3675, 0.3032, 0.3873, 0.4495, 0.1289]))]))
proj_v.load_state_dict(OrderedDict([('weight', torch.tensor([[ 0.2719, -0.0664,  0.3742,  0.3409],
                      [ 0.1177,  0.4588, -0.3498, -0.4507],
                      [-0.4851, -0.3594,  0.2124, -0.1817],
                      [-0.3162, -0.1503, -0.1955,  0.3816],
                      [ 0.0005, -0.0776, -0.4964, -0.1608],
                      [ 0.0581, -0.1783, -0.2951,  0.0964],
                      [-0.3771,  0.1194, -0.4692,  0.1051],
                      [-0.4773, -0.0826,  0.4722, -0.2247],
                      [-0.1419,  0.0064,  0.3859,  0.1678],
                      [ 0.2845, -0.4944,  0.4023, -0.2722]])),
('bias', torch.tensor([ 0.1750,  0.1422, -0.3162,  0.2938,  0.0050, -0.0249,  0.2706, -0.2545, 0.0081, -0.2179]))]))
reproj.load_state_dict(OrderedDict([('weight',
    torch.tensor([[ 0.2090, -0.1364,  0.2534,  0.0466, -0.1310,  0.1216,  0.0299, -0.2586, 0.3095, -0.2708],
            [-0.2939,  0.2829, -0.0410,  0.2643,  0.0008, -0.3008,  0.2150,  0.0737, -0.0611, -0.0701],
            [ 0.2135, -0.1298, -0.3017,  0.2684, -0.0917, -0.1428,  0.1363, -0.1012, 0.2472,  0.2877],
            [ 0.1128,  0.0359, -0.1215,  0.2214,  0.2173, -0.1789,  0.1038,  0.0059, 0.2911, -0.0398]])),
             ('bias', torch.tensor([ 0.2898,  0.0237, -0.2518, -0.2610]))]))
sequence = torch.tensor([[
                   [ 0.1177,  0.4588, -0.3498, -0.4507],
                   [-0.0764,  0.4670, -0.2218, -0.3888],
                   [-0.3771,  0.1194, -0.4692,  0.1051],
                   [-0.1419,  0.0064,  0.3859,  0.1678],
                   [-0.3162, -0.1503, -0.1955,  0.3816],
                   ]])
assert torch.norm(transformer_head(sequence, sequence, sequence, proj_q, proj_k, proj_v, reproj, softmax) -\
    torch.tensor([[[ 0.3581,  0.3061, -0.1407, -0.0638],
                   [ 0.3595,  0.3037, -0.1386, -0.0639],
                   [ 0.3618,  0.3005, -0.1352, -0.0635],
                   [ 0.3583,  0.3030, -0.1361, -0.0625],
                   [ 0.3604,  0.3014, -0.1349, -0.0626]]])).item() < 1e-3

## 3. Geração de Texto com Transformer

Nesta seção, você implementará um modelo de linguagem Transformer do tipo *decoder-only* (estilo GPT), que então será treinado no conjunto de dados TinyShakespeare, que tem 40.000 linhas de texto extraídas de obras de Shakespeare.

Você vai usar blocos de Transformer prontos do PyTorch ao invés de criar a arquitetura do zero. A tarefa a ser aprendida pelo modelo é a predição do próximo *token*, a exemplo do é feito nos Grandes Modelos de Linguagem (*Large Language Models* - LLMs). No caso, vamos considerar que um *token* é um caractere (diferentemente da escolha usual de um *token* ser uma palavra). Também adota-se as seguintes técnicas usuais de Transformer: *encoder* posicional e máscara causal.

In [ ]:
import math
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(1337)
print("Using device:", device)
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))

O conjunto de dados TinyShakespeare é um arquivo de texto de aproximadamente 1 MB. Como estamos modelando o problema no nível de caractere, o vocabulário é o conjunto de caracteres distinto do texto (apenas 65 diferentes caracteres).

A célula abaixo carrega o conjunto de dados e faz uma breve exploração.

In [ ]:
!wget -q -nc https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(f"Length of dataset in characters: {len(text):,}")
print("---- first 250 characters ----")
print(text[:250])
print("---- Number of unique characters in the dataset:", len(set(text)))

A célula abaixo apresenta os caracteres presentes no conjunto de dados e gera comandos para codificar e decodificar uma cadeia de caracteres.

In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

encode = lambda s: [stoi[c] for c in s]          # str  -> list[int]
decode = lambda l: "".join(itos[i] for i in l)   # list[int] -> str

print("Unique characters in the dataset:", "".join(chars))
print("Encoding and decoding test:")
print("Hello", "->", encode("Hello"), "->", decode(encode("Hello")))

Abaixo, divimos o conjunto de dados em treino (`train`), validação (`val`) e test (`test`) em proporções 80%, 10% e 10%, respectivamente.

A classe `CharDataset` facilita a manipulação do texto. Basicamente, como o objetivo aqui é predizer o próximo *token*, um item do *dataset* será um par $(x,y)$, em que $x$ é um bloco do texto de tamanho `block_size` e $y$ é obtido aplicando um *shift* para a esquerda de 1 caractere no $x$.

In [ ]:
from torch.utils.data import Dataset, DataLoader

data = torch.tensor(encode(text), dtype=torch.long)
# Splitting the dataset into train, validation, and test sets (80%, 10%, 10%)
n1 = int(0.8 * len(data))
n2 = int(0.9 * len(data))
train_data = data[:n1]
val_data   = data[n1:n2]
test_data  = data[n2:]

class CharDataset(Dataset):
    """
    Sliding-window (x, y) pairs over a 1-D tensor of token ids.

    Each item is a contiguous block of `block_size` tokens (x) and the same
    block shifted right by one (y). `stride` sets the spacing between windows:
    stride=1 -> overlapping windows (uses the full data support); stride=block_size
    -> disjoint windows (each token scored once; cheap, deterministic sweep).
    """
    def __init__(self, data: torch.tensor, block_size: int, stride: int = 1):
        """
        Initializes the CharDataset by storing the input data, block size, and stride as instance variables.
        @param data: tensor of token ids representing the text dataset.
        @param block_size: the size of the contiguous block of tokens to be used as input
        @param stride: the spacing between windows (default is 1 for overlapping windows)
        """
        self.data = data
        self.block_size = block_size
        self.stride = stride

    def __len__(self):
        """
        Returns the number of (x, y) pairs in the dataset.
        @return: integer representing the number of (x, y) pairs in the dataset
        """
        return (len(self.data) - self.block_size - 1) // self.stride + 1

    def __getitem__(self, idx: int):
        """
        Gets the (x, y) pair at the specified index by calculating the starting position based on the index and stride, extracting a chunk of data of size block_size + 1, and returning the input (x) and target (y) tensors.
        @param idx: integer index of the (x, y) pair to retrieve
        @return: tuple of tensors (x, y) where x is the input tensor of
        """
        start = idx * self.stride
        chunk = self.data[start : start + self.block_size + 1]
        return chunk[:-1], chunk[1:]   # x, y (shifted left by one)

# Quick look at one (x, y) pair
xb, yb = CharDataset(train_data, block_size=8)[0]
print("x:", xb.tolist())
print("y:", yb.tolist(), " <- x shifted left by one")

Abaixo, implemente o *encoder* posicional, da mesma forma como é apresentado no artigo de [Vaswani et. al (2017)](https://arxiv.org/abs/1706.03762v5).

Para a posição *pos* e a dimensão *i*, as equações do *encoder* posicional senoidal são:

$$PE_{(pos,\,2i)} = \sin\!\left(\frac{pos}{10000^{2i/d_{model}}}\right), \qquad
  PE_{(pos,\,2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

`forward(x)` recebe `x` com *shape* `(B, T, d_model)` e deve adicionar o *encoding* para as primeiras `T` posições.

<details><summary><b>---Dica---</b></summary>
<p>
Para implementar o fator de divisão de uma forma numericamente robusta, use:

`div_factor = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000) / d_model))`

O *encoder* posicional usado no artigo original é fixo. Para indicar ao PyTorch que ele não deve tentar otimizar os vetores definidos pelo *encoder* posicional, faça:

`self.register_buffer("pe", pe.unsqueeze(0))`
</p>
</details>

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000):
        """
        Implements the positional encoding module (PE) by building a matrix of sinusoids of shape (max_len, d_model) and
        registering it as a buffer in the module.
        @param d_model: the dimension of the model (embedding size)
        @param max_len: the maximum length of the input sequences (number of positions)
        """
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(max_len).unsqueeze(1).float()
        # Implement the remaining code here!
        raise NotImplementedError() # Delete this line
        self.register_buffer("pe", pe.unsqueeze(0)) # Keep this line as is, it registers the positional encoding as a non-trainable buffer in the module.

    def forward(self, x: torch.tensor) -> torch.tensor:
        """
        Does a forward pass through the positional encoding module by adding the positional encoding to the input tensor.
        @param x: (B, T, d_model) tensor of input features
        @return: (B, T, d_model) tensor of input features with positional encoding
        """
        raise NotImplementedError() # Delete this line
        return x

In [ ]:
# Verifying the implementation of the positional encoding
d_model = 32

pos_encoder = PositionalEncoding(32)

assert(pos_encoder.pe.shape == (1, 5000, 32))
assert(pos_encoder.pe[0, 0, 0] == 0.0)
assert(abs(pos_encoder.pe[0, 1, 0].item() - 0.8414709568023682) < 1e-4)
assert(pos_encoder.pe[0, 0, 1] == 1.0)
assert(abs(pos_encoder.pe[0, 1, 1].item() - 0.5403023058681398) < 1e-4)

x = torch.ones(2, 10, d_model)
out = pos_encoder(x)
assert torch.allclose(out[:, :10, :] - 1.0, pos_encoder.pe[:, :10, :], atol=1e-4)

Abaixo, implementamos o modelo do Transformer. Perceba que embora usemos `nn.TransformerEncoderLayer`, estamos implementando apenas um *decoder*.

Também, para evitar que o modelo use informação do futuro para prever o próximo *token*, **implemente** uma máscara causal no método `generate_causal_mask()`. Como visto em sala, a máscara é definida através de uma matriz $\mathbf{M} \in \mathbb{R}^{n \times n}$ com 0's e $-\infty$'s tal que:
\begin{equation}
    M_{i,j} =
\left\{
\begin{array}{l}
    0, j \leq i, \\
    -\infty, \text{otherwise}.
\end{array}
\right.
\end{equation}

<details><summary><b>---Dica---</b></summary>
<p>
As seguintes funções podem te ajudar:

`torch.full`: Define uma matriz cheia com base em algum valor.

`torch.triu`: Define uma matriz triangular superior. Através do parâmetro `diagonal`, você pode alterar qual a diagonal que será considerada na definição do triângulo de 0's.

Para definir infinito no Python, use `-math.inf`.

Sugiro testar os comandos numa célula de rascunho antes de implementar no modelo.
</p>
</details>

In [ ]:
from math import inf

class SimpleTransformer(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, nhead: int, num_layers: int,
                 dim_feedforward: int = 512, max_len: int = 512, dropout: float = 0.1):
        """
        Implements a simple Transformer-based language model using PyTorch's built-in TransformerEncoder and TransformerEncoderLayer modules.
        @param vocab_size: the size of the vocabulary (number of unique tokens)
        @param d_model: the dimension of the model (embedding size)
        @param nhead: the number of attention heads in the multi-head attention mechanism
        @param num_layers: the number of layers in the Transformer encoder
        @param dim_feedforward: the dimension of the feedforward network in each Transformer encoder layer
        @param max_len: the maximum length of input sequences (used for positional encoding)
        @param dropout: the dropout rate to be applied in the Transformer encoder layers
        """
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,   # tensors are (batch, seq, feature)
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc_out = nn.Linear(d_model, vocab_size)

    def generate_causal_mask(self, size: int, device):
        """
        Generates a causal mask of shape (size, size) where the upper triangle (excluding the diagonal) is filled with -inf and the rest is filled with 0. This mask is used to prevent the model from attending to future tokens during training.
        @param size: the size of the mask (number of tokens in the sequence)
        @param device: the device on which to create the mask
        @return: a tensor of shape (size, size) containing the causal mask
        """
        raise NotImplementedError() # Delete this line
        return torch.full((size, size), 0.0, device=device) # Change this line

    def forward(self, x: torch.tensor) -> torch.tensor:
        """
        Forward pass through the transformer.
        @param x: (B, T) tensor of token ids
        @return: (B, T, vocab_size) tensor of logits for the next token
        """
        mask = self.generate_causal_mask(x.size(1), x.device)
        emb = self.pos_encoder(self.embedding(x) * math.sqrt(self.d_model))
        out = self.transformer(emb, mask=mask)
        return self.fc_out(out)

A seguir, define-se os hiperparâmetros e verifica-se a implementação.

In [ ]:
# --- model size ---
d_model         = 256
nhead           = 8
num_layers      = 6
dim_feedforward = 1024
dropout         = 0.2

# --- sequence / optimization ---
block_size    = 128     # context length (how many chars the model sees)
batch_size    = 64
max_iters     = 5000
eval_interval = 500
eval_iters    = 200
learning_rate = 3e-4

model = SimpleTransformer(
    vocab_size=vocab_size, d_model=d_model, nhead=nhead, num_layers=num_layers,
    dim_feedforward=dim_feedforward, max_len=block_size, dropout=dropout,
).to(device)

print("Number of parameters in the model:", sum(p.numel() for p in model.parameters()))

mask = model.generate_causal_mask(4, device)  # quick test of the mask shape and values
print("Causal mask for size 4:\n", mask)
assert(mask == torch.tensor([[0., -inf, -inf, -inf],
                             [0.,   0., -inf, -inf],
                             [0.,   0.,   0., -inf],
                             [0.,   0.,   0.,   0.]], device=device)).all()

Na célula seguinte, treinamos no modelo.

Como o *dataset* é grande, decidimos por treinar o modelo por apenas 1 época e avaliá-lo a cada 500 iterações (antes de completar uma época).

Como o *dataset* é grande, o treinamento pode demorar bastante tempo se você não tiver uma boa GPU. Assim, se estiver treinando no Colab, eu recomendo usar apenas 5000 iterações. Caso tenha uma boa GPU, pode treinar usando todo o *dataset*.

In [ ]:
# Create the dataloaders
train_loader      = DataLoader(CharDataset(train_data, block_size, stride=1),
                               batch_size=batch_size, shuffle=True,  drop_last=True)
val_loader        = DataLoader(CharDataset(val_data,   block_size, stride=block_size),
                               batch_size=batch_size, shuffle=False)
test_loader       = DataLoader(CharDataset(test_data,  block_size, stride=block_size),
                               batch_size=batch_size, shuffle=False)

# Function used to evaluate the model on a given dataloader (train, val, or test)
@torch.no_grad()
def evaluate(loader):
    model.eval()
    total, n = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = F.cross_entropy(logits.view(-1, vocab_size), yb.view(-1))
        total += loss.item() * xb.size(0)
        n += xb.size(0)
    model.train()
    return total / n

model = SimpleTransformer(
    vocab_size=vocab_size, d_model=d_model, nhead=nhead, num_layers=num_layers,
    dim_feedforward=dim_feedforward, max_len=block_size, dropout=dropout,
).to(device)

num_epochs = 1   # A single shuffled pass over the training windows (since the dataset is big)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# max_iters = len(train_loader) * num_epochs # If you have a good GPU, I recommend training with the whole training set
max_iters = 5000 # If you are training on Colab or a bad GPU, use 5000 iters so you do not have to wait too long

history = dict(train_loss=[], val_loss=[])
step = 0
train_loss = 0.0
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        logits = model(xb)
        loss = F.cross_entropy(logits.view(-1, vocab_size), yb.view(-1))
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        step += 1

        if step % eval_interval == 0:
            print(f"steps {step:>5}/{max_iters}: train loss {train_loss / eval_interval:.4f}, "
                  f"val loss {evaluate(val_loader):.4f}")
            history["train_loss"].append(train_loss / eval_interval)
            history["val_loss"].append(evaluate(val_loader))
            train_loss = 0.0

        if step == max_iters:
            break


print(f"final: train loss {evaluate(train_loader):.4f}, "
      f"val loss {evaluate(val_loader):.4f}, test loss {evaluate(test_loader):.4f}")
print("Training complete!")

A seguir, mostra-se as curvas de aprendizado (de treino e de validação).

<font color='red'>**Observando as curvas obtidas, discuta no seu relatório o resultado em termos de qualidade do aprendizado. Inclua o gráfico no seu relatório!**</font>

In [ ]:
import matplotlib.pyplot as plt

eval_steps = list(range(eval_interval, eval_interval * len(history["train_loss"]) + 1, eval_interval))
plt.plot(eval_steps, history["train_loss"], label="train loss")
plt.plot(eval_steps, history["val_loss"], label="val loss")
plt.xlabel("Evaluation step")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid()
plt.show()
plt.savefig("loss_plot.png", dpi=400)

Finalmente, usamos o modelo para gerar um texto.

Para amostrar, passamos o contexto atual pelo modelo, pegamos os *logits* na última posição, convertemos em uma distribuição de probabilidade e amostramos um novo *token*. Anexamos esse *token* e repetimos. Recortamos o contexto para os últimos `block_size` *tokens*, pois é tudo o que o *encoder* posicional do modelo cobre.

O parâmetro `temperature` controla a aleatoriedade (valores menores deixam o modelo mais confiante/repetitivo); o `top_k` restringe a amostragem aos `k` caracteres mais prováveis.

<font color='red'>**Inclua no seu relatório o texto gerado!**</font>

In [ ]:
@torch.no_grad()
def generate(model, idx, max_new_tokens, temperature=1.0, top_k=None):
    model.eval()
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:] # Crop context
        logits = model(idx_cond)[:, -1, :] # Last-step logits (B, vocab)

        logits = logits / temperature
        if top_k is not None:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = -float("inf")
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)
    return idx

# Start from a single newline character and generate 500 new characters
context = torch.zeros((1, 1), dtype=torch.long, device=device)
out = generate(model, context, max_new_tokens=500, temperature=0.8, top_k=40)
print(decode(out[0].tolist()))

# 4. Entrega

A entrega consiste do notebook no formato **.ipynb** e de um relatório, submetida através do Google Classroom. Modificações nos arquivos do código base são permitidas, desde que o nome e a interface dos scripts “main” não sejam alterados. A princípio, não há limitação de número de páginas para o relatório, mas pede-se que seja sucinto. O relatório deve conter:
- Figuras que comprovem o funcionamento do seu código.
- Demais solicitações feitas ao longo do roteiro.

Por limitações do Google Classroom (e por motivo de facilitar a automatização da correção), entregue seu laboratório com todos os arquivos num único arquivo **.zip** (**não** utilize outras tecnologias de compactação de arquivos) com o seguinte padrão de nome: **“<login_email_google_education>_labX.zip”**. Por exemplo, no meu caso, meu login Google Education é **marcos.maximo**, logo eu entregaria o lab 1 como **“marcos.maximo_lab1.zip”**. **Não** crie subpastas para os arquivos da sua entrega, **deixe todos os arquivos na “raiz” do .zip**.